In [ ]:
# Kaggle-friendly environment adjustments
import os
import sys
import torch

# Use Kaggle working dir and input paths by default
KAGGLE_WORKING = os.environ.get('KAGGLE_WORKING_DIR', '/kaggle/working')
KAGGLE_INPUT = os.environ.get('KAGGLE_INPUT_DIR', '/kaggle/input')

# Results and data directories
RESULTS_DIR = os.path.join(KAGGLE_WORKING, 'results')
DATA_ROOT = KAGGLE_INPUT
os.makedirs(RESULTS_DIR, exist_ok=True)

# Device selection: prefer GPU if available on Kaggle
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Kaggle-friendly notebook:')
print('  RESULTS_DIR ->', RESULTS_DIR)
print('  DATA_ROOT   ->', DATA_ROOT)
print('  DEVICE      ->', DEVICE)

# NOTE: If your dataset is in a named Kaggle dataset under /kaggle/input/<dataset-name>,
# set the environment variable KAGGLE_INPUT_DIR to that path or update DATA_ROOT below.
# Example: os.environ['KAGGLE_INPUT_DIR'] = '/kaggle/input/my-dataset-name'

# Optional: install missing packages (uncomment if needed)
# !pip install some-package


# NB10 -- Final Benchmarking: FLAIR vs Continual Learning Baselines

## Shared Autonomy -- Incremental Learning of Joint-Space Policies

This notebook runs the **final comparison** of our proposed FLAIR
(FiLM-based Learning with Adaptive Importance and Replay) against 5 established continual
learning baselines on the sequential shared-autonomy task.

### Strategies Compared

| # | Strategy | Type | Key Mechanism |
|---|----------|------|---------------|
| 1 | Joint Training  | Upper bound | Retrains on all data |
| 2 | Online EWC | Regularization | Fisher-weighted penalty |
| 3 | A-GEM | Memory | Gradient projection |
| 4 | DER++ | Replay | Dark knowledge distillation |
| 5 | **FLAIR (ours)** | **Hybrid** | **FiLM + Replay + Regularization + Multi-Head** |

### FLAIR Architecture

```
Input -> [Shared Layer 1] -> ReLU -> FiLM(task) -> Dropout
      -> [Shared Layer 2] -> ReLU -> FiLM(task) -> Dropout
      -> [Shared Head + Task Head (blended)] -> Output
```

**Key features:**
1. **FiLM Conditioning** -- per-task feature modulation (gamma * h + beta)
2. **Task-Aware Replay** -- DER++-style buffer with correct FiLM routing
3. **Importance Regularization** -- Fisher-based backbone weight protection
4. **RetroBoost** -- retroactive importance boosting across tasks
5. **FiLM Warm-Start** -- initialize new task FiLM from most similar existing task
6. **Adaptive Replay** -- forgetting-weighted replay sampling
7. **Multi-Head Output** -- per-task output heads blended with shared head


In [ ]:
import os, sys, abc, copy, json, logging, random, time, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader, TensorDataset

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING, format='%(message)s')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} -- device: {DEVICE}')
if DEVICE == 'cuda': print(f'  GPU: {torch.cuda.get_device_name()}')

PROJECT_ROOT = "/home/g0amer/Desktop/thesis/phd_project/thesis_project"
HARMONIC_DIR = f'{PROJECT_ROOT}/data/processed/harmonic'
RESULTS_DIR  = f'{PROJECT_ROOT}/experiments/exp07_flair'
os.makedirs(RESULTS_DIR, exist_ok=True)

participants = sorted([d for d in os.listdir(HARMONIC_DIR)
                       if os.path.isdir(os.path.join(HARMONIC_DIR, d)) and d.startswith('p')])
print(f'Available participants: {len(participants)} -- {participants}')
print(f'Results: {RESULTS_DIR}')


In [ ]:
# ================================================================
# Continual Learning Metrics: ACC, F, BWT, FWT, Memory, Time
# (Identical to NB08 for fair comparison)
# ================================================================

@dataclass
class TaskResult:
    task_id: int; loss: float; mse: float; mae: float; r2: float
    n_samples: int = 0
    per_dim_mse: list = field(default_factory=list)
    ss_res: float = 0.0
    ss_tot: float = 0.0


class CLMetrics:
    # Tracks the R2 accuracy matrix a[i][j] = R2 on task j after training up to task i
    def __init__(self):
        self.R = []       # accuracy matrix (R2): R[i][j]
        self.MSE = []     # mse matrix
        self.per_dim = [] # per-dim MSE matrix
        self._last_results = []
        self.task_names = []
        self._random_baselines = []  # b_j for FWT

    def set_random_baselines(self, baselines):
        self._random_baselines = baselines

    def record(self, trained_up_to, task_results):
        r2_row = [r.r2 for r in task_results]
        mse_row = [r.mse for r in task_results]
        pdim_row = [r.per_dim_mse for r in task_results]
        while len(self.R) <= trained_up_to:
            self.R.append([])
            self.MSE.append([])
            self.per_dim.append([])
        self.R[trained_up_to] = r2_row
        self.MSE[trained_up_to] = mse_row
        self.per_dim[trained_up_to] = pdim_row
        self._last_results = task_results

    @property
    def T(self):
        return len(self.R)

    @property
    def ACC(self):
        if self.T == 0: return float('nan')
        last = self.R[-1]
        return float(np.mean(last)) if last else float('nan')

    @property
    def forgetting(self):
        T = self.T
        if T < 2: return 0.0
        fgt_sum = 0.0
        for j in range(T - 1):
            max_prev = max(self.R[l][j] for l in range(T - 1)
                          if j < len(self.R[l]))
            final = self.R[T - 1][j] if j < len(self.R[T - 1]) else 0.0
            fgt_sum += max(0.0, max_prev - final)
        return fgt_sum / (T - 1)

    @property
    def BWT(self):
        T = self.T
        if T < 2: return 0.0
        bwt = 0.0
        cnt = 0
        for j in range(T - 1):
            if j < len(self.R[T - 1]) and j < len(self.R[j]):
                bwt += self.R[T - 1][j] - self.R[j][j]
                cnt += 1
        return bwt / cnt if cnt > 0 else 0.0

    @property
    def FWT(self):
        T = self.T
        if T < 2: return 0.0
        fwt = 0.0
        cnt = 0
        for j in range(1, T):
            if j < len(self.R[j - 1]):
                b_j = self._random_baselines[j] if j < len(self._random_baselines) else 0.0
                fwt += self.R[j - 1][j] - b_j
                cnt += 1
        return fwt / cnt if cnt > 0 else 0.0

    @property
    def overall_r2(self):
        if not self._last_results: return float('nan')
        ss_res = sum(r.ss_res for r in self._last_results)
        ss_tot = sum(r.ss_tot for r in self._last_results)
        return 1 - ss_res / max(ss_tot, 1e-8)

    def summary_dict(self):
        return {
            'ACC': self.ACC, 'F': self.forgetting,
            'BWT': self.BWT, 'FWT': self.FWT,
            'overall_r2': self.overall_r2,
            'r2_matrix': self.R, 'mse_matrix': self.MSE,
            'per_dim_matrix': self.per_dim,
            'task_names': self.task_names,
        }


@dataclass
class TaskData:
    task_id: int; participant_id: str
    obs_train: torch.Tensor; act_train: torch.Tensor
    obs_val: torch.Tensor; act_val: torch.Tensor
    obs_test: torch.Tensor; act_test: torch.Tensor

    @property
    def n_train(self): return self.obs_train.shape[0]
    @property
    def n_val(self): return self.obs_val.shape[0]
    @property
    def n_test(self): return self.obs_test.shape[0]
    def train_loader(self, bs=256):
        return DataLoader(TensorDataset(self.obs_train, self.act_train), batch_size=bs, shuffle=True)
    def val_loader(self, bs=512):
        return DataLoader(TensorDataset(self.obs_val, self.act_val), batch_size=bs)
    def test_loader(self, bs=512):
        return DataLoader(TensorDataset(self.obs_test, self.act_test), batch_size=bs)

print('CLMetrics (ACC, F, BWT, FWT) & TaskData defined')


In [ ]:
# ================================================================
# Models & CL Strategies (Baselines + FLAIR)